In [1]:
import os
import requests
import pandas as pd
from datetime import datetime, date

def fetch_starting_pitchers(target_date):
    date_str = target_date.isoformat()
    url = f'https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={date_str}&hydrate=probablePitcher(note,stats,person)'

    response = requests.get(url)
    if response.status_code != 200:
        print(f"❌ Failed to fetch data: {response.status_code}")
        return None

    data = response.json()
    games = data.get('dates', [])[0].get('games', []) if data.get('dates') else []

    pitcher_data = []
    for game in games:
        home_team = game['teams']['home']['team']['name']
        away_team = game['teams']['away']['team']['name']

        home_pitcher = game['teams']['home'].get('probablePitcher', {}).get('fullName', 'TBD')
        away_pitcher = game['teams']['away'].get('probablePitcher', {}).get('fullName', 'TBD')

        pitcher_data.append({
            'Date': date_str,
            'Away Team': away_team,
            'Away Starter': away_pitcher,
            'Home Team': home_team,
            'Home Starter': home_pitcher
        })

    return pd.DataFrame(pitcher_data)

# ==== MAIN EXECUTION ====
today = date.today()
df = fetch_starting_pitchers(today)

if df is not None:
    # Output directory and file
    folder = "starting-pitchers"
    os.makedirs(folder, exist_ok=True)
    filename = f"starting_pitchers_{today.isoformat()}.csv"
    filepath = os.path.join(folder, filename)

    df.to_csv(filepath, index=False)
    print(f"✅ Starting pitchers saved to {filepath}")
    print(df.head())


✅ Starting pitchers saved to starting-pitchers/starting_pitchers_2025-05-28.csv
         Date             Away Team     Away Starter            Home Team  \
0  2025-05-28   Los Angeles Dodgers  Clayton Kershaw  Cleveland Guardians   
1  2025-05-28  San Francisco Giants     Landen Roupp       Detroit Tigers   
2  2025-05-28       Minnesota Twins      Pablo López       Tampa Bay Rays   
3  2025-05-28     Chicago White Sox      Shane Smith        New York Mets   
4  2025-05-28        Boston Red Sox     Brayan Bello    Milwaukee Brewers   

      Home Starter  
0     Kolby Allard  
1     Jackson Jobe  
2   Drew Rasmussen  
3  Griffin Canning  
4   Freddy Peralta  
